# EDA dan Prioritas Penghapusan File Duplikat

Notebook ini mengolah `hasil_deteksi_nama_file_identik.csv` untuk:

- melakukan EDA dan pemeriksaan kualitas data;
- menganalisis ekstensi, folder, ukuran, dan kelompok nama duplikat;
- memberi rekomendasi file untuk ditinjau atau dihapus;
- memprioritaskan arsip `.zip`, `.rar`, `.7z`, `.tar`, `.gz`, `.bz2`, dan `.xz`;
- mengekspor rekomendasi ke CSV.

**Notebook ini tidak menghapus file.** Kolom `rekomendasi` hanya menjadi bahan review manual.

## 1. Konfigurasi dan pembacaan data

In [ ]:
from pathlib import Path
import re
import pandas as pd
import matplotlib.pyplot as plt

INPUT_CSV = Path.cwd() / "hasil_deteksi_nama_file_identik.csv"
INPUT_SEP = ","
OUTPUT_SEP = "|"

# Parameter utama rekomendasi. Silakan ubah sesuai kebutuhan.
MIN_FILE_SIZE_MB = 100
MIN_COPIES = 2
ONLY_MATCH_CRITERIA = True
OUTPUT_RECOMMENDATION = Path.cwd() / "rekomendasi_prioritas_penghapusan.csv"
OUTPUT_SUMMARY = Path.cwd() / "ringkasan_eda_file_duplikat.csv"

if not INPUT_CSV.exists():
    raise FileNotFoundError(f"CSV tidak ditemukan: {INPUT_CSV.resolve()}")

required_columns = {"nama_file", "path_lengkap", "ukuran_byte", "ukuran_MB"}
df = pd.read_csv(INPUT_CSV, sep=INPUT_SEP, encoding="utf-8-sig")
missing_columns = required_columns.difference(df.columns)
if missing_columns:
    raise ValueError(f"Kolom wajib tidak ditemukan: {sorted(missing_columns)}")

df["ukuran_byte"] = pd.to_numeric(df["ukuran_byte"], errors="coerce")
df["ukuran_MB"] = pd.to_numeric(df["ukuran_MB"], errors="coerce")
df["nama_file"] = df["nama_file"].fillna("").astype(str)
df["path_lengkap"] = df["path_lengkap"].fillna("").astype(str)
df.head()

## 2. EDA dan kualitas data

In [ ]:
quality_report = pd.DataFrame({
    "baris": [len(df)],
    "nama_kosong": [df["nama_file"].eq("").sum()],
    "path_kosong": [df["path_lengkap"].eq("").sum()],
    "ukuran_tidak_valid": [df["ukuran_byte"].isna().sum()],
    "nama_unik": [df["nama_file"].str.casefold().nunique()],
    "total_ukuran_GB": [df["ukuran_byte"].sum() / 1024**3],
})
display(quality_report.round(2))

df["ekstensi"] = df["nama_file"].str.extract(r"(\.[^.\\]+)$", expand=False).fillna("[tanpa ekstensi]").str.lower()
df["nama_key"] = df["nama_file"].str.casefold()
df["jumlah_dengan_nama_sama"] = df.groupby("nama_key")["nama_key"].transform("size")
df["folder"] = df["path_lengkap"].map(lambda value: str(Path(value).parent) if value else "[tidak diketahui]")

print("Ekstensi terbanyak:")
display(df["ekstensi"].value_counts().head(20).rename_axis("ekstensi").reset_index(name="jumlah"))
print("Kelompok nama dengan anggota terbanyak:")
display(df["jumlah_dengan_nama_sama"].value_counts().sort_index().rename_axis("jumlah_file_per_nama").reset_index(name="jumlah_kelompok"))

## 3. Visualisasi

In [ ]:
top_extensions = df.groupby("ekstensi")["ukuran_byte"].sum().sort_values(ascending=False).head(15).sort_values()
top_extensions.plot(kind="barh", figsize=(10, 6), title="Total ukuran berdasarkan ekstensi")
plt.xlabel("Ukuran (byte)")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 5))
plt.hist(df["ukuran_MB"].dropna(), bins=40)
plt.title("Distribusi ukuran file duplikat")
plt.xlabel("Ukuran (MB)")
plt.ylabel("Jumlah file")
plt.tight_layout()
plt.show()

## 4. Parameter custom dan rekomendasi

Kriteria utama rekomendasi adalah file berukuran minimal `MIN_FILE_SIZE_MB`
dan memiliki minimal `MIN_COPIES` file dengan nama sama. Perhitungan
jumlah mencakup file di lokasi berbeda karena dikelompokkan berdasarkan
nama file, bukan path. Parameter dapat diubah pada cell konfigurasi.

- `archive`: +60 untuk arsip umum, +80 untuk `.zip` atau `.rar`;
- `duplicate_count`: +10 per salinan tambahan, maksimum +30;
- `size`: +10 untuk file minimal 1 GB, +20 untuk minimal 5 GB;
- `sort`: ukuran byte menurun, lalu jumlah file dengan nama sama menurun;
- `recommendation`: `TINJAU PRIORITAS TINGGI`, `UKURAN BESAR, TUNGGAL`, atau `DUPLIKAT KECIL`.

`ONLY_MATCH_CRITERIA = True` membuat output hanya berisi file yang memenuhi
kedua kriteria. Ubah menjadi `False` untuk menampilkan semua file.

In [ ]:
archive_extensions = {".zip", ".rar", ".7z", ".tar", ".gz", ".bz2", ".xz", ".tgz", ".tbz2"}
high_priority_archives = {".zip", ".rar"}

df["is_archive"] = df["ekstensi"].isin(archive_extensions)
df["archive_score"] = df["ekstensi"].map(
    lambda extension: 80 if extension in high_priority_archives else 60 if extension in archive_extensions else 0
)
df["duplicate_score"] = ((df["jumlah_dengan_nama_sama"] - 1).clip(lower=0) * 10).clip(upper=30)
df["size_score"] = 0
df.loc[df["ukuran_MB"] >= MIN_FILE_SIZE_MB, "size_score"] = 10
df.loc[df["ukuran_MB"] >= 1024, "size_score"] = 20
# Skor membantu label review; urutan utama tetap ukuran, lalu jumlah duplikat.
df["priority_score"] = df[["size_score", "duplicate_score", "archive_score"]].sum(axis=1)
df["memenuhi_kriteria"] = (
    (df["ukuran_MB"] >= MIN_FILE_SIZE_MB)
    & (df["jumlah_dengan_nama_sama"] >= MIN_COPIES)
)
df["rekomendasi"] = "REFERENSI"
df.loc[df["memenuhi_kriteria"], "rekomendasi"] = "TINJAU PRIORITAS TINGGI"
df.loc[(~df["memenuhi_kriteria"]) & (df["ukuran_MB"] >= MIN_FILE_SIZE_MB), "rekomendasi"] = "UKURAN BESAR, TUNGGAL"
df.loc[(~df["memenuhi_kriteria"]) & (df["jumlah_dengan_nama_sama"] >= MIN_COPIES), "rekomendasi"] = "DUPLIKAT KECIL"

recommendation_columns = [
    "rekomendasi", "priority_score", "nama_file", "ekstensi", "path_lengkap",
    "jumlah_dengan_nama_sama", "ukuran_byte", "ukuran_MB", "is_archive",
    "memenuhi_kriteria"
]
recommendations = df[recommendation_columns].copy()
if ONLY_MATCH_CRITERIA:
    recommendations = recommendations[recommendations["memenuhi_kriteria"]]
recommendations = recommendations.sort_values(
    ["ukuran_byte", "jumlah_dengan_nama_sama", "priority_score"],
    ascending=[False, False, False]
).reset_index(drop=True)
display(recommendations.head(50))

## 5. Ekspor hasil

File hasil hanya berisi rekomendasi. Tidak ada file pada `path_lengkap` yang
dihapus atau dipindahkan oleh notebook ini.

In [ ]:
recommendations.to_csv(OUTPUT_RECOMMENDATION, sep=OUTPUT_SEP, index=False, encoding="utf-8-sig")
summary = pd.DataFrame({
    "rekomendasi": recommendations["rekomendasi"].value_counts().index,
    "jumlah_file": recommendations["rekomendasi"].value_counts().values,
})
summary.to_csv(OUTPUT_SUMMARY, sep=OUTPUT_SEP, index=False, encoding="utf-8-sig")

print(f"Rekomendasi disimpan: {OUTPUT_RECOMMENDATION.resolve()}")
print(f"Ringkasan disimpan: {OUTPUT_SUMMARY.resolve()}")
display(summary)
